In [1]:
import wget
import tarfile
import os
import time
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from matplotlib import rc
import sys
import numpy.core.numeric as numeric
import numpy as np

In [3]:
df_sm = pd.read_pickle("/home/jpittard/depot/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_pickle/events_0.pkl.gz")
df_eft_lin = pd.read_pickle("/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_lin_pickles_15/tt_ctGRe/events_0.pkl.gz")
df_eft_quad1 = pd.read_pickle("/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_lin_quad_pickles_15/tt_ctGRe/events_0.pkl.gz")
#df_eft_quad2 = pd.read_pickle("/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_quad_pickles_15/tt_ctGRe/events_0.pkl.gz")

df_eft_quad1

,gen_ttbar_mass,gen_b1k,gen_b2k,gen_b1r,gen_b2r,gen_b1n,gen_b2n,gen_c_nn,gen_c_rr,gen_c_kk,gen_ll_cHel,gen_ll_cLab,gen_llbar_delta_phi,gen_ttbar_rapidity,gen_b1q,gen_llbar_pt,gen_l_eta
0,391.055817,0.907170,-0.038775,0.273120,0.811421,-0.320075,0.583175,-0.186660,0.221615,-0.035176,0.000220,0.551972,0.661791,-1.353950,-0.273120,76.544373,-2.133993
1,1067.822144,0.313013,0.211856,-0.636944,0.742252,0.704504,0.635750,0.447888,-0.472773,0.066314,-0.041429,-0.679028,3.069330,-0.697487,-0.636944,67.868729,-0.009563
2,491.312042,0.767301,0.501544,-0.583778,-0.091281,0.265428,0.860303,0.228349,0.053288,0.384835,-0.666472,-0.555831,2.398268,-0.760492,-0.583778,35.185181,0.333495
3,413.167969,0.574432,0.846855,0.547135,0.528620,0.608827,-0.058283,-0.035484,0.289227,0.486461,-0.740203,0.246412,2.273892,1.136909,0.547135,43.231583,0.495447
4,489.231934,0.191838,-0.081052,0.930524,0.419673,-0.311967,0.904049,-0.282034,0.390516,-0.015549,-0.092933,-0.666802,2.189799,-0.183511,-0.930524,74.145844,-0.527787
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14999995,451.433716,-0.249931,-0.309183,0.228413,0.374616,-0.940937,0.874110,-0.822482,0.085567,0.077274,0.659641,0.731652,0.799645,0.925094,-0.228413,82.082458,1.439075
14999996,401.645721,-0.631555,-0.310824,0.138043,-0.319007,-0.762944,-0.895334,0.683089,-0.044037,0.196302,-0.835355,0.263891,2.734939,0.921060,0.138043,12.417736,1.049103
14999997,406.949463,0.433818,-0.978782,0.057657,0.180266,-0.899154,-0.097418,0.087593,0.010394,-0.424613,0.326626,0.544932,1.346415,-0.141070,-0.057657,70.637367,0.739788
14999998,380.978729,-0.853899,-0.444627,0.292814,-0.172076,0.430252,0.879032,0.378205,-0.050386,0.379666,-0.707485,0.407577,2.220373,1.245780,-0.292814,44.805454,1.006614


In [4]:
def event_weights_lin_quad(structure_constants, wc_values):
    """
    Computes the total event weight along with linear and quadratic contributions.

    Parameters:
    structure_constants (list): Structure constants [s_0, s_1, ..., s_15, s_11, s_22, ..., s_ij] for a single event.
    wc_values (list): Arbitrary Wilson coefficient values [c1, c2, ..., cN].

    Returns:
    tuple: (total_weight, linear_weight, quadratic_weight)
    """
    num_WCs = len(wc_values)  # Number of Wilson coefficients

    # Construct linear terms (without the constant term)
    linear_terms = wc_values  # (c1, c2, ..., cN)
    s_linear = structure_constants[:, 1:len(linear_terms) + 1]   # Skip s0 and match the rest to the linear terms
    w_linear = np.dot(s_linear, linear_terms)

    # Construct quadratic terms (without the constant term)
    quadratic_terms = [wc_values[k]**2 for k in range(num_WCs)]  # c1^2, c2^2, ..., cN^2
    quadratic_terms.extend([wc_values[k1] * wc_values[k2] for k1, k2 in combinations(range(num_WCs), 2)])  # c1*c2, etc.

    s_quad = structure_constants[:,len(linear_terms) + 1 :]  # Skip s0 and linear terms, match to the quadratic terms
    w_quad = np.dot(s_quad, quadratic_terms)
    sm = structure_constants[:,0]
    # Add the structure constant for the constant term separately
    w_linear_with_sm= sm + w_linear  # Adding s0 to the linear contribution
    w_quad_with_sm= sm + w_quad  # Adding s0 to the quadratic contribution

    # Compute total weight
    total_weight = w_linear_with_sm + w_quad_with_sm - sm

    return sm, w_linear_with_sm, w_quad_with_sm, total_weight

In [5]:
M = 15000000
wc_dim = 16

theta_2 = [0.0] * wc_dim
theta_2[0] = 15.0

In [8]:
sm, w_linear_with_sm, w_quad_with_sm, total_weight = event_weights_lin_quad(df_sm['gen_c_kk'], theta_2)

KeyError: 'key of type tuple not found and not a MultiIndex'

In [2]:
M = 15000000
wc_dim = 16

# WC names (optional mapping)
wc_name = ['ctGRe', 'ctGIm', 'cQj18', 'cQj38', 'cQj11', 'cQj31', 'ctu8', 'ctd8',
           'ctj8', 'cQu8', 'cQd8', 'ctu1', 'ctd1', 'ctj1', 'cQu1', 'cQd1']

    max_events=M

SyntaxError: incomplete input (2974210049.py, line 24)